**Система:** каталог файлов  
**Протокол:** TCP/IP  
**Поля записи:** номер файла, имя файла, дата создания, количество обращений  
**Основные операции:** добавление, удаление, сумма обращений, сортированный отчет и просмотр списка

Проект состоит из серверной и клиентской частей. Обмен выполняется по TCP-сокету,
а команды передаются строками с разделителем `,`.

## 0. Импорт библиотек

In [ ]:
import socket
import threading
from datetime import datetime

## 1. Серверная часть

In [ ]:
class FileCatalogServer:
    def __init__(self, host="127.0.0.1", port=12345):
        self.host = host
        self.port = port
        self.files = {}
        self.next_id = 1
        self.running = False
        self.server_socket = None

    def process_command(self, data):
        try:
            parts = data.split(",")
            command = parts[0].strip()

            if command == "+":
                return self.add_file(parts[1:])
            if command == "-":
                return self.delete_file(parts[1:])
            if command == "sum":
                return self.calculate_sum()
            if command == "report":
                return self.generate_report()
            if command == "list":
                return self.list_files()

            return "ERROR: Неизвестная команда"
        except Exception as e:
            return f"ERROR: {e}"

    def add_file(self, data):
        if len(data) < 2:
            return "ERROR: Недостаточно данных"

        try:
            file_name = data[0].strip()
            access_count = int(data[1].strip())

            if not file_name:
                return "ERROR: Имя файла не может быть пустым"

            if access_count < 0:
                return "ERROR: Количество обращений не может быть отрицательным"

            self.files[self.next_id] = {
                "file_name": file_name,
                "creation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "access_count": access_count,
            }

            result = f"OK: Файл добавлен с ID {self.next_id}"
            self.next_id += 1
            return result

        except ValueError:
            return "ERROR: Количество обращений должно быть числом"

    def delete_file(self, data):
        if not data:
            return "ERROR: Не указан ID файла"

        try:
            file_id = int(data[0].strip())

            if file_id in self.files:
                del self.files[file_id]
                return f"OK: Файл с ID {file_id} удален"

            return f"ERROR: Файл с ID {file_id} не найден"

        except ValueError:
            return "ERROR: ID файла должен быть числом"

    def calculate_sum(self):
        total = sum(
            file_data["access_count"]
            for file_data in self.files.values()
        )
        return f"SUM: {total}"

    def generate_report(self):
        if not self.files:
            return "REPORT: Нет данных для отчета"

        sorted_files = sorted(
            self.files.items(),
            key=lambda item: item[1]["access_count"]
        )

        lines = [
            "ОТЧЕТ",
            "ID | Имя файла | Дата создания | Обращений",
            "-" * 65,
        ]

        for file_id, file_data in sorted_files:
            lines.append(
                f"{file_id:2} | "
                f"{file_data['file_name']:20} | "
                f"{file_data['creation_date']:19} | "
                f"{file_data['access_count']:8}"
            )

        return "\\n".join(lines)

    def list_files(self):
        if not self.files:
            return "LIST: Нет файлов в каталоге"

        lines = ["ФАЙЛЫ В КАТАЛОГЕ:"]
        for file_id, file_data in self.files.items():
            lines.append(
                f"ID {file_id}: "
                f"{file_data['file_name']} "
                f"({file_data['access_count']} обращений)"
            )

        return "\\n".join(lines)

    def handle_client(self, client_socket, address):
        print(f"Подключение от {address}")

        with client_socket:
            while self.running:
                data = client_socket.recv(4096)

                if not data:
                    break

                request = data.decode("utf-8")
                response = self.process_command(request)
                client_socket.sendall(response.encode("utf-8"))

        print(f"Отключение от {address}")

    def serve_forever(self):
        self.server_socket = socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        )
        self.server_socket.setsockopt(
            socket.SOL_SOCKET,
            socket.SO_REUSEADDR,
            1
        )
        self.server_socket.bind((self.host, self.port))
        self.server_socket.listen(5)
        self.running = True

        print(f"Сервер запущен на {self.host}:{self.port}")

        try:
            while self.running:
                client_socket, address = self.server_socket.accept()
                thread = threading.Thread(
                    target=self.handle_client,
                    args=(client_socket, address),
                    daemon=True
                )
                thread.start()
        finally:
            self.server_socket.close()

    def stop(self):
        self.running = False

        if self.server_socket is not None:
            try:
                self.server_socket.shutdown(socket.SHUT_RDWR)
            except OSError:
                pass

            try:
                self.server_socket.close()
            except OSError:
                pass

## 2. Клиентская часть

In [ ]:
class FileCatalogClient:
    def __init__(self, host="127.0.0.1", port=12345):
        self.host = host
        self.port = port

    def send_command(self, command):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client:
            client.connect((self.host, self.port))
            client.sendall(command.encode("utf-8"))
            return client.recv(4096).decode("utf-8")

    def add_file(self, file_name, access_count):
        if not file_name.strip():
            return "ERROR: Имя файла не может быть пустым"

        access_count = int(access_count)

        if access_count < 0:
            return "ERROR: Количество обращений не может быть отрицательным"

        return self.send_command(
            f"+,{file_name.strip()},{access_count}"
        )

    def delete_file(self, file_id):
        return self.send_command(f"- ,{int(file_id)}".replace("- ", "-"))

    def calculate_sum(self):
        return self.send_command("sum")

    def generate_report(self):
        return self.send_command("report")

    def list_files(self):
        return self.send_command("list")

## 3. Проверка обработчика команд без сетевого соединения

In [ ]:
demo_server = FileCatalogServer()

commands = [
    "+,report.pdf,12",
    "+,data.csv,7",
    "+,photo.jpg,21",
    "list",
    "sum",
    "report",
    "-,2",
    "list",
]

for command in commands:
    print(f"> {command}")
    print(demo_server.process_command(command))
    print()

## 4. Запуск TCP-сервера в отдельном потоке

In [ ]:
server = FileCatalogServer()

server_thread = threading.Thread(
    target=server.serve_forever,
    daemon=True
)

server_thread.start()

## 5. Проверка клиента

In [ ]:
client = FileCatalogClient()

print(client.add_file("document.txt", 15))
print(client.add_file("archive.zip", 4))
print(client.add_file("image.png", 27))

print()
print(client.list_files())

print()
print(client.calculate_sum())

print()
print(client.generate_report())

## 6. Остановка сервера

In [ ]:
server.stop()
print("Сервер остановлен.")

## Итог

In [ ]:
print("Поддерживаемые команды:")
print("+      — добавить файл")
print("-      — удалить файл")
print("sum    — сумма количества обращений")
print("report — отчет с сортировкой")
print("list   — список файлов")